# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [17]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [4]:
df = spaceship.copy()

# Drop missing values
df = df.dropna()

# Simplify Cabin: keep only the deck letter
df["Cabin"] = df["Cabin"].str.split("/").str[0]

# Drop identifier columns
df = df.drop(columns=["PassengerId", "Name"])

# Convert boolean columns into numeric format
df["CryoSleep"] = df["CryoSleep"].astype(int)
df["VIP"] = df["VIP"].astype(int)
df["Transported"] = df["Transported"].astype(int)

X = df.drop(columns=["Transported"])
y = df["Transported"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=17, stratify=y)

In [5]:
def fit_encoders(df):
    encoder_home = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    encoder_cabin = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    encoder_dest = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    
    encoder_home.fit(df[["HomePlanet"]])
    encoder_cabin.fit(df[["Cabin"]])
    encoder_dest.fit(df[["Destination"]])
    
    return encoder_home, encoder_cabin, encoder_dest


def OHE(df, encoder_home, encoder_cabin, encoder_dest):
    df = df.reset_index(drop=True).copy()
    
    home_enc = encoder_home.transform(df[["HomePlanet"]])
    df_home = pd.DataFrame(home_enc, columns=encoder_home.get_feature_names_out(["HomePlanet"]))
    
    cabin_enc = encoder_cabin.transform(df[["Cabin"]])
    df_cabin = pd.DataFrame(cabin_enc, columns=encoder_cabin.get_feature_names_out(["Cabin"]))
    
    dest_enc = encoder_dest.transform(df[["Destination"]])
    df_dest = pd.DataFrame(dest_enc, columns=encoder_dest.get_feature_names_out(["Destination"]))
    
    df = pd.concat([df, df_home, df_cabin, df_dest], axis=1)
    df = df.drop(columns=["HomePlanet", "Cabin", "Destination"])
    
    return df

In [6]:
encoder_home, encoder_cabin, encoder_dest = fit_encoders(X_train)

X_train = OHE(X_train, encoder_home, encoder_cabin, encoder_dest)
X_test = OHE(X_test, encoder_home, encoder_cabin, encoder_dest)

X_test = X_test[X_train.columns]

In [7]:
X_test = X_test[X_train.columns]

Now perform the same as before:
- Feature Scaling
- Feature Selection


In [8]:
normalizer = MinMaxScaler()

normalizer.fit(X_train)

X_train_norm = normalizer.transform(X_train)
X_test_norm = normalizer.transform(X_test)

In [9]:
X_train_norm = pd.DataFrame(X_train_norm, columns=X_train.columns, index=X_train.index)
X_test_norm = pd.DataFrame(X_test_norm, columns=X_test.columns, index=X_test.index)

In [10]:
X_train_norm.head()

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,...,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
0,0.0,0.544304,0.0,0.013105,0.000000,0.071003,0.001383,0.001279,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.0,0.316456,0.0,0.000000,0.051510,0.000000,0.066182,0.000000,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,1.0,0.430380,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.455696,0.0,0.000000,0.010136,0.000000,0.000000,0.031078,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,0.0,0.151899,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


In [13]:
#Feature selection

selector = SelectKBest(score_func=f_classif, k=10)

selector.fit(X_train_norm, y_train)

X_train_selected = selector.transform(X_train_norm)
X_test_selected = selector.transform(X_test_norm)

selected_features = X_train_norm.columns[selector.get_support()]

selected_features

Index(['CryoSleep', 'RoomService', 'Spa', 'VRDeck', 'HomePlanet_Earth',
       'HomePlanet_Europa', 'Cabin_B', 'Cabin_C', 'Destination_55 Cancri e',
       'Destination_TRAPPIST-1e'],
      dtype='object')

In [14]:
X_train_selected = pd.DataFrame(X_train_selected, columns=selected_features, index=X_train_norm.index)
X_test_selected = pd.DataFrame(X_test_selected, columns=selected_features, index=X_test_norm.index)

In [15]:
feature_scores = pd.DataFrame({
    "feature": X_train_norm.columns,
    "score": selector.scores_
}).sort_values(by="score", ascending=False)

feature_scores

,feature,score
0,CryoSleep,1522.160524
3,RoomService,348.242843
6,Spa,276.466738
7,VRDeck,238.577293
9,HomePlanet_Europa,177.753816
8,HomePlanet_Earth,158.365270
12,Cabin_B,109.479498
19,Destination_55 Cancri e,78.428132
21,Destination_TRAPPIST-1e,61.991950
13,Cabin_C,56.952278


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.
- Evaluate your model

In [18]:
gb_base = GradientBoostingClassifier(random_state=17)

gb_base.fit(X_train_selected, y_train)

y_pred_base = gb_base.predict(X_test_selected)

In [ ]:
base_results = { "model": "Gradient Boosting Base", "accuracy": accuracy_score(y_test, y_pred_base), "precision": precision_score(y_test, y_pred_base), 
                "recall": recall_score(y_test, y_pred_base), "f1_score": f1_score(y_test, y_pred_base)}

base_results

{'model': 'Gradient Boosting Base',
 'accuracy': 0.7806354009077155,
 'precision': 0.7589531680440771,
 'recall': 0.8273273273273273,
 'f1_score': 0.7916666666666666}

**Grid/Random Search**
- Define hyperparameters to fine tune.
- Run Grid Search

In [21]:
param_grid = {"n_estimators": [50, 100, 150], "learning_rate": [0.01, 0.05, 0.1], "max_depth": [2, 3, 4],
    "min_samples_split": [2, 5], "min_samples_leaf": [1, 2], "subsample": [0.8, 1.0]}

In [22]:
gb_model = GradientBoostingClassifier(random_state=17)

grid_search = GridSearchCV(estimator=gb_model, param_grid=param_grid, scoring="f1", cv=5, n_jobs=-1, verbose=1)

grid_search.fit(X_train_selected, y_train)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


,estimator,GradientBoost...ndom_state=17)
,param_grid,"{'learning_rate': [0.01, 0.05, ...], 'max_depth': [2, 3, ...], 'min_samples_leaf': [1, 2], 'min_samples_split': [2, 5], ...}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [23]:
print("Best parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation F1-score:")
print(grid_search.best_score_)

Best parameters:
{'learning_rate': 0.05, 'max_depth': 4, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 150, 'subsample': 0.8}

Best cross-validation F1-score:
0.8039373823146485


In [24]:
best_gb_model = grid_search.best_estimator_

y_pred_tuned = best_gb_model.predict(X_test_selected)

In [25]:
tuned_results = {"model": "Gradient Boosting Tuned", "accuracy": accuracy_score(y_test, y_pred_tuned), "precision": precision_score(y_test, y_pred_tuned),
    "recall": recall_score(y_test, y_pred_tuned), "f1_score": f1_score(y_test, y_pred_tuned)}

tuned_results

{'model': 'Gradient Boosting Tuned',
 'accuracy': 0.7821482602118003,
 'precision': 0.7560975609756098,
 'recall': 0.8378378378378378,
 'f1_score': 0.7948717948717948}

- Evaluate your model

In [26]:
comparison_df = pd.DataFrame([base_results, tuned_results])

comparison_df

,model,accuracy,precision,recall,f1_score
0,Gradient Boosting Base,0.780635,0.758953,0.827327,0.791667
1,Gradient Boosting Tuned,0.782148,0.756098,0.837838,0.794872


In [27]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))

Confusion Matrix:
[[476 180]
 [108 558]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.73      0.77       656
           1       0.76      0.84      0.79       666

    accuracy                           0.78      1322
   macro avg       0.79      0.78      0.78      1322
weighted avg       0.79      0.78      0.78      1322



### Tuned Model Evaluation

After applying Grid Search, the **Gradient Boosting Tuned** model performed slightly better than the baseline model.

The baseline Gradient Boosting model achieved an F1-score of **0.7917**, while the tuned model achieved an F1-score of **0.7949**. This means that hyperparameter tuning produced a small improvement in the balance between precision and recall.

The tuned model also achieved a slightly higher accuracy, increasing from **0.7806** to **0.7821**. Its recall also improved from **0.8273** to **0.8378**, meaning the tuned model became better at identifying passengers who were actually transported.

However, the improvement is very small. Precision decreased slightly, from **0.7590** to **0.7561**, which means the tuned model made a few more false positive predictions.

Overall, the **Gradient Boosting Tuned** model is the best final model because it has the highest F1-score and recall. However, the difference compared to the baseline model is marginal, so the baseline Gradient Boosting model was already performing close to its optimal configuration.